In [1]:
import os, json, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers
from transformers import AutoModelForMaskedLM

In [2]:
model_name = 'facebook/esm2_t33_650M_UR50D'

from go_ml.train_utils import get_elm_df
elm_df = get_elm_df(instance_url='/home/andrew/GO_interp/data/elm/elm_instances_all.tsv',
                cls_url='/home/andrew/GO_interp/data/elm/elm_classes.tsv',
                sequence_url='/home/andrew/GO_interp/data/elm/elm_sequences_all.fasta')
elm_id_counter = Counter(elm_df['ELMIdentifier'])
elm_df = elm_df[[elm_id_counter[eid] >= 10 for eid in elm_df['ELMIdentifier']]]
elm_cls_group = elm_df.groupby('Primary_Acc')[['ELMIdentifier']].agg(list).reset_index()
elm_df_cls = elm_cls_group.merge(elm_df[['Primary_Acc', 'Sequence']].drop_duplicates('Primary_Acc'), on='Primary_Acc', how='left')
cls_list = elm_df['ELMIdentifier'].unique(); cls_list.sort()
cls_ind = {cls_str:i for i, cls_str in enumerate(cls_list)}
elm_df_cls['ELMInd'] = elm_df_cls['ELMIdentifier'].apply(lambda l: [cls_ind[c] for c in l])
label_series = [np.zeros(len(cls_list)) for _ in range(len(elm_df_cls))]
for t_m, elm_ind in zip(label_series, elm_df_cls['ELMInd']):
    t_m[elm_ind] = 1
elm_df_cls['ELMLabel'] = label_series

In [5]:
train_path = "/home/andrew/GO_interp/data/elm"
prot_sequences, seq_ids = load_protein_sequences(f"{train_path}/lig_elm_instances.fasta")
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
ps, si = [], []
for pid, pseq in zip(seq_ids, prot_sequences):
    if(len(pseq) <= 800):
        ps.append(pseq)
        si.append(pid)
prot_sequences = ps; seq_ids = si

In [6]:
dataset = ProtDataset(seq_ids, prot_sequences)
SEQUENCE_MASK_TOKEN = tokenizer.get_vocab()['<mask>']
i2t = {i:t for t, i in tokenizer.get_vocab().items()}
base_tokens = torch.tensor([range(4, 24)]).long().flatten()
aa_list = [i2t[i] for i in range(4, 24)]

from go_ml.train_utils import bert_mask
from go_ml.data_utils import collate_dict
from torch.utils.data import DataLoader

def seq_collator(data_dict_list):
    sample = collate_dict(data_dict_list)
    inputs = tokenizer.batch_encode_plus(sample["seq"],
                                                add_special_tokens=True,
                                                return_attention_mask=True, 
                                                padding=True)
    sample['seq_ind'] = torch.tensor(inputs['input_ids'])
    sample['masked_ind'] = bert_mask(sample['seq_ind'], sequence_mask_token=SEQUENCE_MASK_TOKEN, base_tokens=base_tokens, mut_per=0.15)
    sample['mask'] = torch.BoolTensor(inputs['attention_mask'])
    return sample
dataloader = DataLoader(dataset, collate_fn=seq_collator, batch_size=4, shuffle=True)

In [ ]:
from go_ml.train_utils import bert_mask
from go_ml.data_utils import collate_dict
from torch.utils.data import DataLoader

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
prot_ids = elm_df_cls['Primary_Acc']
prot_sequences = elm_df_cls['Sequence']
prot_labels = [{'elm_label': x} for x in elm_df_cls['ELMLabel']] 

dataset = ProtDataset(prot_ids, prot_sequences, prot_data=prot_labels)
SEQUENCE_MASK_TOKEN = tokenizer.get_vocab()['<mask>']
i2t = {i:t for t, i in tokenizer.get_vocab().items()}
base_tokens = torch.tensor([range(4, 24)]).long().flatten()
aa_list = [i2t[i] for i in range(4, 24)]

def joint_mask_seq_collator(data_dict_list):
    sample = collate_dict(data_dict_list)
    inputs = tokenizer.batch_encode_plus(sample["seq"],
                                                add_special_tokens=True,
                                                return_attention_mask=True, 
                                                padding=True)
    sample['seq_ind'] = torch.tensor(inputs['input_ids'])
    sample['masked_ind'] = bert_mask(sample['seq_ind'], sequence_mask_token=SEQUENCE_MASK_TOKEN, base_tokens=base_tokens, mut_per=0.15)
    sample['mask'] = torch.BoolTensor(inputs['attention_mask'])
    sample['elm_label'] = torch.LongTensor(np.stack(sample['elm_label']))
    return sample
dataloader = DataLoader(dataset, collate_fn=joint_mask_seq_collator, batch_size=4, shuffle=True)

In [8]:
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

class ClassCondBert(pl.LightningModule):
    def __init__(self, model_args) -> None:
        super(ClassCondBert, self).__init__()
        self.model_name = model_args['model_name']
        self.model = AutoModelForMaskedLM.from_pretrained(self.model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.classification_head = nn.Linear(2*model_args['hidden_dim'], model_args['num_cls'])

    def forward(self, input_ids, attention_mask):
        logits, hidden_states = self.model(input_ids, attention_mask, output_hidden_states=True).values()
        word_embeddings = hidden_states[-1]
        pooling = self.pool_strategy({"token_embeddings": word_embeddings,
                                      "cls_token_embeddings": word_embeddings[:, 0],
                                      "attention_mask": attention_mask,
                                      })
        return logits, self.classification_head(pooling)
    
    def pool_strategy(self, features, pool_cls=True, pool_mean=True):
        token_embeddings = features['token_embeddings']
        cls_token = features['cls_token_embeddings']
        attention_mask = features['attention_mask']

        ## Pooling strategy
        output_vectors = []
        if pool_cls:
            output_vectors.append(cls_token)
        if pool_mean:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = input_mask_expanded.sum(1)
            sum_mask = torch.clamp(sum_mask, min=1e-9)
            output_vectors.append(sum_embeddings / sum_mask)

        # output_vector = torch.stack(output_vectors, -1).sum(dim=-1)
        output_vector = torch.cat(output_vectors, 1)
        return output_vector

    def training_step(self, batch, batch_idx):
        seq_logits, cls_logits = self.forward(batch['masked_ind'], batch['mask'])
        seq_ind = batch['seq_ind']
        seq_ce = F.cross_entropy(seq_logits.reshape(-1, seq_logits.shape[-1]), seq_ind.flatten(), reduce=False).reshape(seq_ind.shape)
        mask = (batch['masked_ind'] != batch['seq_ind'])
        mask_loss = (seq_ce*mask).sum() / mask.sum()

        labels = batch['elm_label'].float()
        cls_loss = F.binary_cross_entropy_with_logits(cls_logits, labels)
        loss = mask_loss + cls_loss
        self.log('train_mask_loss', mask_loss)
        self.log('train_cls_loss', cls_loss) 
        self.log('train_loss', loss)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        return optimizer
    
masked_train_model = ClassCondBert({'model_name': model_name, "hidden_dim": 1280, "num_cls": len(elm_df_cls['ELMLabel'][0])})

In [9]:
trainer = pl.Trainer(devices=[0], max_epochs=3)
trainer.fit(masked_train_model, train_dataloaders=dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]

  | Name                | Type           | Params | Mode 
---------------------------------------------------------------
0 | model               | EsmForMaskedLM | 652 M  | eval 
1 | classification_head | Linear         | 250 K  | train
---------------------------------------------------------------
652 M     Trainable params
0         Non-trainable params
652 M     Total params
2,610.430 Total estimated model params size (MB)
/home/andrew/anaconda3/envs/esm/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

dict_keys(['prot_id', 'seq', 'elm_label'])


/home/andrew/anaconda3/envs/esm/lib/python3.10/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys(['prot_id', 'seq', 'elm_label'])
dict_keys([

/home/andrew/anaconda3/envs/esm/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [7]:
from transformers import AutoModel, AutoTokenizer
from torch.nn.functional import cross_entropy

class MaskedObjFinetune(pl.LightningModule):
    def __init__(self, model_args) -> None:
        super(MaskedObjFinetune, self).__init__()
        self.model_name = model_args['model_name']
        self.model = AutoModelForMaskedLM.from_pretrained(self.model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

    def training_step(self, batch, batch_idx):
        model_output = self.model.forward(batch['masked_ind'], batch['mask'])
        logits = model_output.logits
        seq_ind = batch['seq_ind']
        seq_ce = cross_entropy(logits.reshape(-1, logits.shape[-1]), seq_ind.flatten(), reduce=False).reshape(seq_ind.shape)
        mask = (batch['masked_ind'] != batch['seq_ind'])
        loss = (seq_ce*mask).sum() / mask.sum()
        self.log('train_loss', loss)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        return optimizer

masked_train_model = MaskedObjFinetune({'model_name': model_name})

In [12]:
trainer = pl.Trainer(devices=[0], max_epochs=3)
trainer.fit(masked_train_model, train_dataloaders=dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]

  | Name  | Type           | Params | Mode
------------------------------------------------
0 | model | EsmForMaskedLM | 652 M  | eval
------------------------------------------------
652 M     Trainable params
0         Non-trainable params
652 M     Total params
2,609.426 Total estimated model params size (MB)


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=3` reached.
